# Sesi 7 · Dasar pandas: Memuat Data BigQuery ke Colab

**Kursus:** Data Analysis dengan BigQuery, Python & BI
**Nama:** *tulis namamu di sini*
**Tanggal:** 23 November 2026

Hari ini SQL dan Python bertemu. Kita menjalankan **tabel analisis** dari sesi 4 di BigQuery, memuat hasilnya ke Colab sebagai **DataFrame**, lalu menjelajahinya dengan **pandas**.

**DataFrame** = tabel di Python: ada baris dan kolom, mirip hasil query atau sheet di spreadsheet. **pandas** = library Python untuk mengolah DataFrame.

> Simpan salinan dulu: **File → Save a copy in Drive**.

## 1. Menghubungkan Colab ke BigQuery

1. Jalankan sel di bawah. Akan muncul jendela login Google: pilih **akun yang sama** dengan BigQuery-mu, lalu klik **Allow/Izinkan**.
2. Ganti `PROJECT_ID` dengan Project ID BigQuery-mu (lihat di bagian atas console BigQuery, misalnya `belajar-data-123456`). Project ID berbeda dengan nama project.

In [ ]:
from google.colab import auth
auth.authenticate_user()   # login dengan akun Google yang sama dengan BigQuery

In [ ]:
import pandas as pd
from google.cloud import bigquery

PROJECT_ID = "isi-project-id-kamu"   # <-- GANTI dengan Project ID-mu
client = bigquery.Client(project=PROJECT_ID)

# Tampilkan angka desimal dengan pemisah ribuan dan 2 angka di belakang koma
pd.set_option("display.float_format", "{:,.2f}".format)

def jalankan_query(sql):
    """Menjalankan query SQL di BigQuery dan mengembalikan hasilnya sebagai DataFrame."""
    return client.query(sql).to_dataframe()

# Uji koneksi
jalankan_query("SELECT 'Terhubung ke BigQuery!' AS pesan")

Kalau muncul error `403 ... Access Denied` atau `Project ... not found`, cek lagi `PROJECT_ID`, dan pastikan kamu login dengan akun Google yang benar.

## 2. Memuat tabel analisis

Ini versi instruktur dari PR sesi 4: satu baris per barang terjual, sudah digabung dengan info produk dan pelanggan. Kamu boleh menggantinya dengan query-mu sendiri, asal nama kolomnya sama.

Query SQL panjang ditulis di antara tiga tanda kutip `"""..."""`.

In [ ]:
SQL_TABEL_ANALISIS = """
WITH item_valid AS (
  SELECT id, order_id, user_id, product_id, status, created_at, sale_price
  FROM `bigquery-public-data.thelook_ecommerce.order_items`
  WHERE status NOT IN ('Cancelled', 'Returned')
    AND DATE(created_at) < DATE_TRUNC(CURRENT_DATE(), MONTH)
)
SELECT
  i.id AS order_item_id,
  i.order_id,
  i.user_id,
  i.product_id,
  i.status,
  DATE(i.created_at) AS order_date,
  DATE_TRUNC(DATE(i.created_at), MONTH) AS order_month,
  i.sale_price,
  p.cost,
  i.sale_price - p.cost AS margin,
  p.category,
  COALESCE(p.brand, 'Tidak diketahui') AS brand,
  p.department,
  u.age,
  CASE
    WHEN u.age < 18 THEN '1. Di bawah 18'
    WHEN u.age < 25 THEN '2. 18-24'
    WHEN u.age < 35 THEN '3. 25-34'
    WHEN u.age < 45 THEN '4. 35-44'
    WHEN u.age < 55 THEN '5. 45-54'
    ELSE '6. 55+'
  END AS age_band,
  u.gender,
  u.country,
  u.traffic_source
FROM item_valid AS i
JOIN `bigquery-public-data.thelook_ecommerce.products` AS p ON i.product_id = p.id
JOIN `bigquery-public-data.thelook_ecommerce.users` AS u ON i.user_id = u.id
"""

df = jalankan_query(SQL_TABEL_ANALISIS)
print("Data berhasil dimuat!")

## 3. Kenalan dengan DataFrame

Lima perintah pertama yang selalu dijalankan analis setiap kali mendapat data baru:

| Perintah | Menjawab |
| --- | --- |
| `df.head()` | Seperti apa isinya? (5 baris pertama) |
| `df.shape` | Berapa baris dan kolom? |
| `df.columns` | Apa saja nama kolomnya? |
| `df.info()` | Apa tipe data setiap kolom? Ada nilai kosong? |
| `df.describe()` | Bagaimana sebaran angka-angkanya? |

In [ ]:
df.head()

In [ ]:
df.shape   # (jumlah baris, jumlah kolom)

In [ ]:
df.columns

In [ ]:
df.info()

Di kolom **Dtype**:

- `int64` / `Int64` = bilangan bulat, `float64` = desimal, `object` = teks.
- `dbdate` = tanggal dari BigQuery. Kita ubah ke `datetime64` supaya bisa diolah pandas (bagian 7).

In [ ]:
df.describe()

Cara membaca `describe()`: `count` = jumlah nilai yang terisi, `mean` = rata-rata, `std` = seberapa menyebar, `min`/`max` = terkecil/terbesar, `25%`/`50%`/`75%` = kuartil (`50%` adalah **median**, nilai tengah).

**Pertanyaan diskusi:** rata-rata `sale_price` lebih besar atau lebih kecil dari mediannya? Apa artinya?

## 4. Memilih kolom

In [ ]:
# Satu kolom -> hasilnya Series (satu kolom berlabel)
df["category"].head()

In [ ]:
# Beberapa kolom -> pakai list di dalam kurung siku: df[[...]]
df[["order_id", "category", "sale_price", "country"]].head()

## 5. Memfilter baris (seperti WHERE)

Tulis kondisi di dalam `df[...]`. Untuk beberapa kondisi:

- `&` = AND, `|` = OR. **Setiap kondisi wajib diberi kurung** `( )`.
- `.isin([...])` = IN di SQL.

In [ ]:
# Barang yang dibeli pelanggan dari Jepang
jepang = df[df["country"] == "Japan"]
jepang.shape

In [ ]:
# Barang Jeans dengan harga di atas 100 USD
df[(df["category"] == "Jeans") & (df["sale_price"] > 100)].head()

In [ ]:
# Pelanggan dari Jepang atau Prancis yang datang dari Facebook
df[df["country"].isin(["Japan", "France"]) & (df["traffic_source"] == "Facebook")].head()

## 6. Mengurutkan, menghitung, dan mengelompokkan

| SQL | pandas |
| --- | --- |
| `ORDER BY kolom DESC` | `df.sort_values("kolom", ascending=False)` |
| `GROUP BY x` + `COUNT(*)` | `df["x"].value_counts()` |
| `GROUP BY x` + `SUM(y)` | `df.groupby("x")["y"].sum()` |
| `COUNT(DISTINCT x)` | `df["x"].nunique()` |

In [ ]:
# 5 barang termahal yang terjual
df.sort_values("sale_price", ascending=False).head()

In [ ]:
# Jumlah barang terjual per negara
df["country"].value_counts()

In [ ]:
# Persentase, bukan jumlah: normalize=True
df["traffic_source"].value_counts(normalize=True) * 100

In [ ]:
# Angka-angka utama bisnis
pendapatan = df["sale_price"].sum()
jumlah_pesanan = df["order_id"].nunique()
jumlah_pelanggan = df["user_id"].nunique()

print(f"Pendapatan       : {pendapatan:,.0f} USD")
print(f"Jumlah pesanan   : {jumlah_pesanan:,}")
print(f"Jumlah pelanggan : {jumlah_pelanggan:,}")
print(f"AOV              : {pendapatan / jumlah_pesanan:,.2f} USD")

In [ ]:
# Pendapatan per kategori, dari terbesar
df.groupby("category")["sale_price"].sum().sort_values(ascending=False)

### groupby + agg: beberapa ringkasan sekaligus

Format `nama_kolom_baru=("kolom_sumber", "fungsi")`. Fungsi yang umum: `"sum"`, `"mean"`, `"median"`, `"count"`, `"nunique"`, `"min"`, `"max"`.

In [ ]:
per_kategori = (
    df.groupby("category")
      .agg(pendapatan=("sale_price", "sum"),
           margin=("margin", "sum"),
           pesanan=("order_id", "nunique"),
           harga_rata=("sale_price", "mean"))
      .sort_values("pendapatan", ascending=False)
)
per_kategori.head(10)

## 7. Membuat kolom baru

In [ ]:
# Kolom hasil hitungan
per_kategori["margin_persen"] = per_kategori["margin"] / per_kategori["pendapatan"] * 100
per_kategori.sort_values("margin_persen", ascending=False).head()

In [ ]:
# Mengubah kolom tanggal ke tipe datetime pandas
df["order_date"] = pd.to_datetime(df["order_date"])
df["order_month"] = pd.to_datetime(df["order_month"])
df["tahun"] = df["order_date"].dt.year   # .dt untuk mengambil bagian tanggal
df[["order_date", "order_month", "tahun"]].head()

In [ ]:
# Pendapatan per tahun
df.groupby("tahun")["sale_price"].sum()

In [ ]:
# Function dari sesi 6, diterapkan ke setiap baris dengan .apply()
def label_harga(harga):
    """Label harga barang: murah, sedang, atau mahal."""
    if harga > 75:
        return "mahal"
    elif harga >= 25:
        return "sedang"
    else:
        return "murah"

df["label_harga"] = df["sale_price"].apply(label_harga)
df["label_harga"].value_counts()

## 8. Sekilas grafik

pandas bisa langsung membuat grafik sederhana dengan `.plot()`. Di sesi 8 kita buat grafik yang rapi dan siap dipresentasikan.

In [ ]:
pendapatan_bulanan = df.groupby("order_month")["sale_price"].sum()
pendapatan_bulanan.plot(title="Pendapatan bulanan (USD)", figsize=(10, 4));

## 9. Giliran kamu (±10 menit)

**G1.** Berapa jumlah pelanggan unik (`user_id`) dari setiap `gender`?

In [ ]:
# G1

**G2.** Kelompok umur (`age_band`) mana yang paling banyak menyumbang pendapatan? Tampilkan pendapatan dan jumlah pesanan unik per `age_band`.

In [ ]:
# G2

## 10. PR sesi 7: jelajahi tabel users

Muat tabel `users` dengan query di bawah, lalu jawab pertanyaan berikut dengan pandas. Tulis jawabanmu (1 kalimat) di sel teks di bawah setiap kode.

1. Berapa jumlah baris dan kolom? Kolom mana yang punya nilai kosong?
2. Berapa umur rata-rata, median, termuda, dan tertua pelanggan?
3. 5 negara dengan pelanggan terbanyak, dan berapa persen porsinya?
4. `traffic_source` mana yang paling banyak membawa pelanggan, dan apakah berbeda antara pelanggan pria dan wanita? (Petunjuk: `groupby(["gender", "traffic_source"]).size()`)
5. Pada tahun berapa paling banyak pelanggan mendaftar? (Petunjuk: `pd.to_datetime(...)` lalu `.dt.year`)

Unggah notebook ini ke `notebooks/07_pandas_exploration.ipynb`.

In [ ]:
SQL_USERS = """
SELECT id, age, gender, country, city, traffic_source, created_at
FROM `bigquery-public-data.thelook_ecommerce.users`
"""
users = jalankan_query(SQL_USERS)
users.head()

In [ ]:
# PR1

*Jawaban 1:*

In [ ]:
# PR2

*Jawaban 2:*

In [ ]:
# PR3

*Jawaban 3:*

In [ ]:
# PR4

*Jawaban 4:*

In [ ]:
# PR5

*Jawaban 5:*